# Evaluate Rossmann Subsampled Dataset

This notebook demonstrates how to evaluate the `rossmann_subsampled` dataset using the Syntherela benchmark.

## Setup

Install the `syntherela` package and `gdown` to download the data.

In [ ]:
!pip install syntherela gdown

## Download Data

Download and extract the original and synthetic datasets.

In [1]:
import os
import zipfile
import gdown


def download_and_extract(url, filename, extract_path):
    # Create directory
    os.makedirs(extract_path, exist_ok=True)

    # Download
    if not os.path.exists(filename):
        print(f"Downloading {filename}...")
        gdown.download(url, filename, quiet=False)

    # Extract
    print(f"Extracting {filename} to {extract_path}...")
    with zipfile.ZipFile(filename, "r") as zip_ref:
        zip_ref.extractall(extract_path)
    print("Done.")


# URLs for the datasets
orig_url = "https://drive.google.com/uc?id=1FIBnmdQSVUK4xi5uFpzb_vFseK_KLQUG"
synth_url = "https://drive.google.com/uc?id=1VRoU57Z-J2QV9J4QTNWo-XTWdD8hqkAl"

# Download and extract to ./data/original and ./data/synthetic
download_and_extract(orig_url, "original.zip", os.path.join("data", "original"))
download_and_extract(synth_url, "synthetic.zip", os.path.join("data", "synthetic"))

Downloading...
From (original): https://drive.google.com/uc?id=1FIBnmdQSVUK4xi5uFpzb_vFseK_KLQUG
From (redirected): https://drive.google.com/uc?id=1FIBnmdQSVUK4xi5uFpzb_vFseK_KLQUG&confirm=t&uuid=742509df-b375-438f-acf3-19a43f12422a
To: c:\Users\valte\Faks\mag2\syntherela\examples\original.zip
100%|██████████| 101M/101M [00:35<00:00, 2.88MB/s] 


Extracting original.zip to data\original...
Done.


Downloading...
From (original): https://drive.google.com/uc?id=1VRoU57Z-J2QV9J4QTNWo-XTWdD8hqkAl
From (redirected): https://drive.google.com/uc?id=1VRoU57Z-J2QV9J4QTNWo-XTWdD8hqkAl&confirm=t&uuid=75236da5-a727-4505-aab8-cad17498fd24
To: c:\Users\valte\Faks\mag2\syntherela\examples\synthetic.zip
100%|██████████| 960M/960M [06:32<00:00, 2.45MB/s] 


Extracting synthetic.zip to data\synthetic...
Done.


## Imports

Import necessary libraries for evaluation.

In [1]:
import os
import json
from xgboost import XGBClassifier

from syntherela.benchmark import Benchmark
from syntherela.metrics.single_column.statistical import (
    ChiSquareTest,
)
from syntherela.metrics.single_table.distance import (
    MaximumMeanDiscrepancy,
)
from syntherela.metrics.multi_table.statistical import CardinalityShapeSimilarity
from syntherela.metrics.multi_table.detection import AggregationDetection
from syntherela.utils import NpEncoder

## Configuration

Set up the dataset, method, and paths.

In [2]:
dataset_name = "rossmann_subsampled"
method = "SDV"  # You can change this to other methods like RCTGAN, MOSTLYAI, etc.
run_id = "1"

# Paths relative to the current working directory
real_data_dir = os.path.join("data", "original")
synthetic_data_dir = os.path.join("data", "synthetic")
results_dir = "results"

# Ensure results directory exists
os.makedirs(results_dir, exist_ok=True)

## Define Metrics

Initialize the metrics to be used for evaluation.

In [3]:
xgb_cls = XGBClassifier
xgb_args = {"seed": 0, "verbosity": 0, "eval_metric": "logloss"}

single_column_metrics = [
    ChiSquareTest(),
]

single_table_metrics = [
    MaximumMeanDiscrepancy(),
]

multi_table_metrics = [
    CardinalityShapeSimilarity(),
    AggregationDetection(
        classifier_cls=xgb_cls, classifier_args=xgb_args, random_state=42
    ),
]

## Run Benchmark

Initialize the `Benchmark` class and run the evaluation.

In [4]:
benchmark = Benchmark(
    real_data_dir=real_data_dir,
    synthetic_data_dir=synthetic_data_dir,
    results_dir=results_dir,
    benchmark_name="ExampleBenchmark",
    single_column_metrics=single_column_metrics,
    single_table_metrics=single_table_metrics,
    multi_table_metrics=multi_table_metrics,
    run_id=run_id,
    sample_id="sample1",
    datasets=[dataset_name],
    methods=[method],
)

# Run the benchmark
benchmark.run()

Starting benchmark for rossmann_subsampled, method_name SDV


Running Single Column Metrics: 100%|██████████| 19/19 [00:00<00:00, 702.12it/s]


Generating report ...

(1/1) Evaluating Column Shapes: |██████████| 19/19 [00:00<00:00, 81.31it/s]|
Column Shapes Score: 81.25%

Overall Score (Average): 81.25%



Running Single Table Metrics: 100%|██████████| 2/2 [02:29<00:00, 74.98s/it]


Generating report ...

(1/1) Evaluating Column Pair Trends: |██████████| 81/81 [00:00<00:00, 84.17it/s]| 
Column Pair Trends Score: 68.06%

Overall Score (Average): 68.06%



Running Multi Table Metrics: 100%|██████████| 2/2 [00:02<00:00,  1.13s/it]


Generating report ...

(1/2) Evaluating Cardinality: |██████████| 1/1 [00:00<00:00, 77.36it/s]|
Cardinality Score: 99.19%

(2/2) Evaluating Intertable Trends: |██████████| 90/90 [00:01<00:00, 56.38it/s]|
Intertable Trends Score: 74.33%

Overall Score (Average): 86.76%

Long Range Scores: {1: 0.7432550594371548}
All avg scores:  0.7432550594371548


## View Results

The results are stored in `benchmark.all_results`. Here we display a summary.

In [5]:
results = benchmark.all_results.get(dataset_name, {}).get(method, {})
with open(
    os.path.join(results_dir, f"{dataset_name}_{method}_{run_id}_sample1.json"), "r"
) as f:
    results = json.load(f)

# Print a few key metrics
print(f"Results for {dataset_name} using {method}:")

if "single_column_metrics" in results:
    print("\nSingle Column Metrics (Chi-Square Test):")
    print(
        json.dumps(
            results["single_column_metrics"].get("ChiSquareTest", {}),
            indent=2,
            cls=NpEncoder,
        )
    )

Results for rossmann_subsampled using SDV:

Single Column Metrics (Chi-Square Test):
{
  "historical": {
    "DayOfWeek": {
      "p_value": 0.0,
      "statistic": 9947.632976704308
    },
    "Open": {
      "p_value": 0.0,
      "statistic": 10790.037302746003
    },
    "Promo": {
      "p_value": 1.4468271126857985e-64,
      "statistic": 287.87116202468746
    },
    "SchoolHoliday": {
      "p_value": 0.0,
      "statistic": 15579.665964401729
    },
    "StateHoliday": {
      "p_value": 1.0,
      "statistic": 0.0
    }
  },
  "store": {
    "Assortment": {
      "p_value": 2.1254423060600378e-72,
      "statistic": 330.06429354165124
    },
    "Promo2": {
      "p_value": 0.13762217723037112,
      "statistic": 2.204346801629361
    },
    "PromoInterval": {
      "p_value": 6.459926417165668e-07,
      "statistic": 28.504955447520626
    },
    "StoreType": {
      "p_value": 7.67914432500336e-97,
      "statistic": 448.28277996055715
    }
  }
}


In [6]:
if "single_table_metrics" in results:
    print("\nSingle Table Metrics (Maximum Mean Discrepancy):")
    print(
        json.dumps(
            results["single_table_metrics"].get("MaximumMeanDiscrepancy", {}),
            indent=2,
            cls=NpEncoder,
        )
    )


Single Table Metrics (Maximum Mean Discrepancy):
{
  "historical": {
    "bootstrap_mean": 1.781512424829418,
    "bootstrap_se": 0.0002764302987081429,
    "reference_ci": [
      0,
      0.01974407448868399
    ],
    "reference_mean": 7.124527525661668e-05,
    "reference_variance": 7.151288965281656e-05,
    "value": 1.7814127858798916
  },
  "store": {
    "bootstrap_mean": 0.01761974308015158,
    "bootstrap_se": 0.0002858398587593915,
    "reference_ci": [
      0,
      0.19432861529272966
    ],
    "reference_mean": 0.010651001547830248,
    "reference_variance": 0.006233947505965626,
    "value": 0.006899605086857402
  }
}


In [7]:
if "multi_table_metrics" in results:
    print("\nMulti Table Metrics:")
    print(json.dumps(results["multi_table_metrics"], indent=2, cls=NpEncoder))


Multi Table Metrics:
{
  "AggregationDetection-XGBClassifier": {
    "store": {
      "SE": 0.0031043349773440224,
      "accuracy": 0.9780269058295964,
      "bin_test_p_val": 0.0,
      "copying_p_val": 1.0
    }
  },
  "CardinalityShapeSimilarity": {
    "store_historical": {
      "pval": 0.9999999999999802,
      "statistic": 0.008071748878923767
    }
  },
  "Trends": {
    "cardinality": 0.9919282511210762,
    "k_hop_similarity": {
      "1": {
        "mean": 0.7432550594371548,
        "se": 0.0235294255643673
      }
    }
  }
}
